# Train Models with Levanter (from README)

This notebook follows the steps from the README to install Levanter and run training commands.

It demonstrates:
- Installing Levanter in editable mode
- Training a GPT-2 "nano" model
- Training a Llama-small model on OpenWebText with configuration overrides

Notes:
- If you use Weights & Biases (W&B), run `wandb login` before training.
- GPU/TPU setup is platform-specific; see docs for GPU/TPU guides.
- This notebook assumes it is located in the `notebooks/` folder of the repo.

## 0) Locate repo (optional)
If running inside a cloned repo, we'll detect the repo root.
Otherwise, we'll install from PyPI/GitHub in the next step.

In [ ]:
import os, sys
from pathlib import Path

def find_repo_root(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src').exists():
            return p
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is not None:
    os.chdir(repo_root)
    print('Using repo root:', os.getcwd())
else:
    print('Repo root not found. Will pip-install Levanter.')


## 1) Install Levanter (editable)
Per the README, after installing JAX for your platform, install Levanter.
If running from this repo, editable install keeps changes live.

In [ ]:
import subprocess, sys, os
from pathlib import Path

def pip(*args):
    print('pip', *args)
    subprocess.check_call([sys.executable, '-m', 'pip', *args])

if (Path.cwd() / 'pyproject.toml').exists():
    # In a cloned repo: editable install
    pip('install', '-e', '.')
else:
    # Not in repo: install package (prefer PyPI, fallback to GitHub)
    try:
        pip('install', 'levanter')
    except subprocess.CalledProcessError:
        pip('install', 'git+https://github.com/stanford-crfm/levanter.git')

# Optional: W&B for logging
# pip('install', 'wandb')
# !wandb login  # or set WANDB_API_KEY env var


## 2) Verify imports and devices

In [ ]:
import jax
import levanter
print('JAX devices:', jax.devices())
print('Levanter version:', getattr(levanter, '__version__', 'dev'))

## 3) Train a GPT-2 "nano" (from README)
This mirrors the README's quickstart. It trains on WikiText-103 by default.

In [ ]:
from pathlib import Path
gpt2_cfg = 'config/gpt2_nano.yaml' if Path('config/gpt2_nano.yaml').exists() else 'gpt2_nano'
cmd = f'python -m levanter.main.train_lm --config_path {gpt2_cfg}'
print(cmd)
!{cmd}


## 4) Train a Llama-small on your own data (from README)
You can override dataset fields via CLI flags. Here we set `--data.id openwebtext`.
Optionally, you may set a tokenizer and/or a cache directory.

In [ ]:
from pathlib import Path
llama_cfg = 'config/llama_small_fast.yaml' if Path('config/llama_small_fast.yaml').exists() else 'llama_small_fast'
cmd = f'python -m levanter.main.train_lm --config_path {llama_cfg} --data.id openwebtext'
print(cmd)
!{cmd}


In [ ]:
# Example with tokenizer and cache dir (edit and uncomment to use):
# !python -m levanter.main.train_lm \
#   --config_path config/llama_small_fast.yaml \
#   --data.id openwebtext \
#   --data.tokenizer "NousResearch/Llama-2-7b-hf" \
#   --data.cache_dir "gs://path/to/cache/dir"

## 5) Customize a config (reference)
Edit YAML under `config/` to change model and training params. For reference, here's the `llama_small_fast.yaml` mentioned in the README.

In [ ]:
from pathlib import Path
print(Path('config/llama_small_fast.yaml').read_text())